# ViT-LENS: Notebook 04 — Feature Discovery & Spatial Localization
**Research Goal**: Discover active dictionary features across dataset images, extract top-activating image patches, generate 14x14 spatial heatmaps, and formulate candidate semantic hypotheses.


In [ ]:
# Step 1: Load Trained SAE & Pretrained ViT
import os, torch
from PIL import Image
import matplotlib.pyplot as plt
from src.utils import get_device, load_config
from src.vit_model import ViTModelWrapper
from src.topk_sae import TopKSparseAutoencoder
from src.dataset import build_dataloaders
from src.feature_explorer import FeatureExplorer
from src.feature_localization import FeatureLocalizer

device = get_device('auto')
config = load_config('config.yaml')

vit = ViTModelWrapper(pretrained=True, target_layer_idx=7, device=device)
sae = TopKSparseAutoencoder(d_in=768, d_sae=3072, top_k=32).to(device)

ckpt_path = os.path.join(config['sae']['checkpoint_dir'], 'sae_checkpoint_latest.pt')
if os.path.exists(ckpt_path):
    ckpt = torch.load(ckpt_path, map_location=device, weights_only=False)
    sae.load_state_dict(ckpt['sae_state_dict'])
    print('Loaded trained SAE checkpoint.')
else:
    print('Using initialized SAE weights for demonstration.')


In [ ]:
# Step 2: Scan Dataset for Top Activating Image Patches
data_dir = config['dataset']['data_dir'] + '/prototype'
eval_loader, _, _ = build_dataloaders(data_dir, batch_size=8, val_split=0.0, num_workers=0)

explorer = FeatureExplorer(sae=sae, vit_wrapper=vit, output_dir=config['feature_discovery']['output_dir'])
feature_records = explorer.scan_dataset_activations(eval_loader, max_samples=40)
print(f'Scanned activations. Found {len(feature_records)} active features.')


In [ ]:
# Step 3: Generate Visual Patch Grids for Discovered Features
target_features = list(feature_records.keys())[:3] if feature_records else [127, 42, 88]
for feat_id in target_features:
    if feat_id in feature_records:
        montage, path = explorer.create_feature_visual_grid(
            feature_id=feat_id,
            records=feature_records[feat_id],
            top_k=8,
            candidate_label=f'Feature #{feat_id}'
        )
        print(f'Saved visual montage for Feature #{feat_id} -> {path}')


In [ ]:
# Step 4: Spatial Localization Heatmap (14x14 -> 224x224 Overlay)
sample_path = './data/samples/sample_retriever.png'
if os.path.exists(sample_path):
    sample_img = Image.open(sample_path)
else:
    sample_img = Image.new('RGB', (224, 224), (200, 150, 100))

localizer = FeatureLocalizer(sae=sae, vit_wrapper=vit)
res = localizer.localize_and_visualize(sample_img, feature_id=127, candidate_label='Fur Texture')

fig, axes = plt.subplots(1, 3, figsize=(12, 4))
axes[0].imshow(res['original_image'])
axes[0].set_title('Original Input')
axes[1].imshow(res['raw_heatmap'])
axes[1].set_title('14x14 Upsampled Heatmap')
axes[2].imshow(res['overlay_image'])
axes[2].set_title(f'Alpha Overlay (Feat #127: {res["candidate_label"]})')
for ax in axes: ax.axis('off')
plt.tight_layout()
plt.show()
